<a href="https://colab.research.google.com/github/Vicodwer/Day---38/blob/main/Day_38.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TMDB Genre Classification (Text Classification)
Dataset: TMDB Top Rated Movies (Page 3)


# Dataset Source
API Link: https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=3

# WORD2VEC TRAINING

In [ ]:
import requests
import pandas as pd
import re
!pip install gensim
from gensim.models import Word2Vec


url = "https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=3"
data = requests.get(url).json()
df = pd.DataFrame(data['results'])

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['clean'] = df['overview'].apply(clean_text)

sentences = [text.split() for text in df['clean']]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1)

# Q1(a) — POLYSEMY DEMONSTRATION

In [ ]:
import collections


all_words = [word for sublist in sentences for word in sublist]


word_counts = collections.Counter(all_words)

for word, count in word_counts.most_common():
    if word in w2v_model.wv.key_to_index:
        most_common_in_vocab.append(word)
    if len(most_common_in_vocab) >= 3:
        break

if len(most_common_in_vocab) < 3:
    print("Warning: Not enough distinct words in vocabulary to perform comparison. Using available words.")

    vocab_list = list(w2v_model.wv.index_to_key)
    base_word = vocab_list[0] if vocab_list else 'unknown'
    word1 = vocab_list[1] if len(vocab_list) > 1 else base_word
    word2 = vocab_list[2] if len(vocab_list) > 2 else base_word
else:
    base_word = most_common_in_vocab[0]
    word1 = most_common_in_vocab[1]
    word2 = most_common_in_vocab[2]

if base_word in w2v_model.wv and word1 in w2v_model.wv and word2 in w2v_model.wv:
    cheap_vec = w2v_model.wv[base_word]

    sim1 = w2v_model.wv.similarity(base_word, word1)
    sim2 = w2v_model.wv.similarity(base_word, word2)

    print(f"cosine({base_word}, {word1}):", sim1)
    print(f"cosine({base_word}, {word2}):", sim2)
else:
    print(f"Could not find suitable words for comparison. Base word: {base_word}, Word 1: {word1}, Word 2: {word2}")

cosine(a, the): 0.01729737
cosine(a, to): -0.016814834


Answer : Word2Vec assigns only one vector to the word “dark,” even though it has multiple meanings (evil vs night). The cosine similarity shows how the same vector relates differently to different context words, but it cannot fully separate meanings.

# Q1(b) — DISAMBIGUATION SYSTEM

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_sentence_vector(sentence, model):
    words = sentence.split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if len(vectors) == 0:

        return np.zeros(model.vector_size)
    return sum(vectors) / len(vectors)


affordable_anchor = ['cheap', 'budget', 'lowcost']
quality_anchor = ['cheap', 'flimsy', 'poor']

def get_anchor_vector(words, model):
    vectors = [model.wv[w] for w in words if w in model.wv]
    if len(vectors) == 0:

        return np.zeros(model.vector_size)
    return sum(vectors) / len(vectors)


anchor1 = get_anchor_vector(affordable_anchor, w2v_model)
anchor2 = get_anchor_vector(quality_anchor, w2v_model)


def disambiguate(sentence):
    sent_vec = get_sentence_vector(sentence, w2v_model)

    sim1 = cosine_similarity([sent_vec], [anchor1])[0][0]
    sim2 = cosine_similarity([sent_vec], [anchor2])[0][0]

    if sim1 > sim2:
        return "Meaning: Affordable"
    else:
        return "Meaning: Low Quality"


print(disambiguate("This movie was made on a cheap budget but looks great"))

Meaning: Low Quality


Answer

The system uses context embeddings to determine meaning. By comparing sentence vectors with predefined anchor vectors, it selects the closest semantic meaning.

# Q1(c) — WINDOW SIZE COMPARISON

Answer

A smaller window size (2) captures syntactic relationships (local context), while a larger window size (10) captures semantic relationships (broader meaning). Therefore, window=10 is better for understanding meaning.

In [ ]:
model_w2 = Word2Vec(sentences, vector_size=100, window=2, min_count=1)
model_w10 = Word2Vec(sentences, vector_size=100, window=10, min_count=1)

print(f"Comparing words for window=2: ({base_word}, {word1})")
print(f"Comparing words for window=10: ({base_word}, {word2})")
print("Window=2 similarity:", model_w2.wv.similarity(base_word, word1))
print("Window=10 similarity:", model_w10.wv.similarity(base_word, word2))

Comparing words for window=2: (a, the)
Comparing words for window=10: (a, to)
Window=2 similarity: 0.00798309
Window=10 similarity: 0.0062303403


Answer

A smaller window size (2) captures syntactic relationships (local context), while a larger window size (10) captures semantic relationships (broader meaning). Therefore, window=10 is better for understanding meaning.

# Q2 — SIMILARITY COMPARISON

In [ ]:
A = "incredible story but terrible ending"
B = "ending was bad but story was amazing"

(1) BOW

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cv = CountVectorizer()
X = cv.fit_transform([A, B]).toarray()

print("BOW similarity:", cosine_similarity([X[0]], [X[1]])[0][0])

BOW similarity: 0.44721359549995787


TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X = tfidf.fit_transform([A, B]).toarray()

print("TF-IDF similarity:", cosine_similarity([X[0]], [X[1]])[0][0])

TF-IDF similarity: 0.29526755538240534


Word2Vec Averaging

In [ ]:
def avg_vector(sentence):
    words = sentence.split()
    vecs = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    return sum(vecs) / len(vecs)

vecA = avg_vector(A)
vecB = avg_vector(B)

print("Word2Vec similarity:", cosine_similarity([vecA], [vecB])[0][0])

Word2Vec similarity: 0.4360573


Sentence-BERT (BEST)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode([A, B])

print("SBERT similarity:", cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT similarity: 0.8665972


Q2(a) ANSWER

Sentence-BERT correctly identifies similarity because it captures full sentence meaning, not just word overlap.

Q2(b) BOW FAILURE

BOW fails because it only counts exact word matches. Words like “terrible” and “bad” or “incredible” and “amazing” are semantically similar but treated as different, reducing similarity.

Q2(c) SEMANTIC GAP

The semantic gap refers to the difference between surface-level word matching and actual meaning. BOW has a large gap, TF-IDF reduces it slightly, Word2Vec captures word-level meaning, and Sentence-BERT captures full sentence semantics, effectively closing the gap.

Git Hub Repository: https://github.com/Vicodwer/Day---38.git